# MU Benchmark & Analysis

**Run on cluster** - Multi-GPU tests require 2+ GPUs

Compares MU implementations:
1. **MU L1 Naive** - Custom naive GEMM (no cuBLAS)
2. **MU L2 Memory** - cuBLAS + fused kernels
3. **MU L3 Compute** - cuBLAS + 8-way ILP
4. **MU L4 Multi-GPU** - Data parallel, sync every iteration
5. **MU L5 Async** - Data parallel, configurable sync interval

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import struct
import re
import os

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../data', exist_ok=True)

# Check GPU count
try:
    result = subprocess.run('nvidia-smi -L | wc -l', 
                           shell=True, capture_output=True, text=True)
    NUM_GPUS = int(result.stdout.strip())
except:
    NUM_GPUS = 1

print(f'Detected {NUM_GPUS} GPU(s)')
print('Ready!')

## 1. Generate Test Matrices

In [ ]:
def generate_lowrank_matrix(m, n, rank, noise_level=0.01, seed=42):
    np.random.seed(seed)
    W_true = np.random.rand(m, rank).astype(np.float32)
    H_true = np.random.rand(rank, n).astype(np.float32)
    X = W_true @ H_true
    X += noise_level * np.random.rand(m, n).astype(np.float32)
    return np.maximum(X, 0).astype(np.float32)

def save_matrix_binary(filename, X):
    m, n = X.shape
    X_col = np.asfortranarray(X)
    with open(filename, 'wb') as f:
        f.write(struct.pack('i', m))
        f.write(struct.pack('i', n))
        f.write(X_col.tobytes())
    print(f'Saved {m}x{n} matrix to {filename}')

In [ ]:
# Configuration - Sizes for A40 GPUs (48GB each)
SIZES = [250, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000, 10000, 12000, 16000, 24000, 32000]
RANK = 20

for size in SIZES:
    filepath = f'../data/lowrank_{size}.bin'
    if not os.path.exists(filepath):
        print(f'Generating {size}x{size}...')
        X = generate_lowrank_matrix(size, size, RANK)
        save_matrix_binary(filepath, X)
    else:
        print(f'{size}x{size} already exists')

print('Done!')

## 2. Run MU Benchmarks (Single GPU)

In [ ]:
def extract_metrics(output):
    """Extract time and error from program output."""
    time_match = re.search(r'Time: ([\d.]+)', output)
    error_match = re.search(r'(?:Final[_ ]?[Ee]rror|error)[:\s]*([\d.e+-]+)', output)
    time = float(time_match.group(1)) if time_match else 0
    error = float(error_match.group(1)) if error_match else 0
    return time, error

# Set up CUDA environment for subprocess calls
CUDA_ENV = os.environ.copy()
cuda_lib_paths = [
    '/sw/pkgs/arc/cuda/12.8.1/lib64',  # Lighthouse cluster
    '/usr/local/cuda/lib64',
    '/usr/local/cuda-12/lib64',
]

for path in cuda_lib_paths:
    if os.path.exists(path):
        current = CUDA_ENV.get('LD_LIBRARY_PATH', '')
        CUDA_ENV['LD_LIBRARY_PATH'] = f"{path}:{current}" if current else path
        print(f"Using CUDA libs: {path}")
        break
else:
    print("WARNING: No CUDA lib path found")

def run_mu(binary, data_file, rank, iters, extra_args=''):
    """Run a MU implementation and return metrics."""
    cmd = f'./{binary} {data_file} {rank} {iters} {extra_args}'
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, 
                               text=True, cwd='..', timeout=600, env=CUDA_ENV)
        time, error = extract_metrics(result.stdout)
        if time == 0 and result.stderr:
            print(f'  STDERR: {result.stderr[:150]}')
        return time, error
    except subprocess.TimeoutExpired:
        print(f'  Timeout after 600s')
        return 0, 0
    except Exception as e:
        print(f'  Error: {e}')
        return 0, 0

In [ ]:
# Run single-GPU MU implementations
MU_ITERS = 100
results = []

print('='*60)
print('Running MU Single-GPU Benchmarks')
print('='*60)

for size in SIZES:
    print(f'\n--- Size: {size}x{size} ---')
    data_file = f'data/lowrank_{size}.bin'
    
    # MU Naive
    print('MU Naive...')
    time, error = run_mu('nmf_naive', data_file, RANK, MU_ITERS)
    if time > 0:
        results.append({'size': size, 'method': 'mu_naive', 'time_ms': time,
                       'iterations': MU_ITERS, 'error': error})
        print(f'  Time: {time:.1f} ms, Error: {error:.4e}')
    
    # MU L2 Memory
    print('MU L2 Memory...')
    time, error = run_mu('nmf_memory_opt', data_file, RANK, MU_ITERS)
    if time > 0:
        results.append({'size': size, 'method': 'mu_l2_memory', 'time_ms': time,
                       'iterations': MU_ITERS, 'error': error})
        print(f'  Time: {time:.1f} ms, Error: {error:.4e}')
    
    # MU L3 Compute
    print('MU L3 Compute...')
    time, error = run_mu('nmf_compute_opt', data_file, RANK, MU_ITERS, '128')
    if time > 0:
        results.append({'size': size, 'method': 'mu_l3_compute', 'time_ms': time,
                       'iterations': MU_ITERS, 'error': error})
        print(f'  Time: {time:.1f} ms, Error: {error:.4e}')

# Create DataFrame
if results:
    mu_df = pd.DataFrame(results)
    mu_df['time_per_iter'] = mu_df['time_ms'] / mu_df['iterations']
    
    print('\n' + '='*60)
    print('Single-GPU Results:')
    print('='*60)
    print(mu_df.to_string(index=False))
    
    # Save to CSV
    mu_df.to_csv('../results/mu_timing.csv', index=False)
    print('\nSaved: results/mu_timing.csv')
else:
    print('\n' + '='*60)
    print('ERROR: No results collected!')
    print('Make sure you ran: make clean && make')
    print('And that data files exist in data/ directory')
    print('='*60)
    mu_df = pd.DataFrame(columns=['size', 'method', 'time_ms', 'iterations', 'error', 'time_per_iter'])

<cell_type>markdown</cell_type>## 3. Run MU Multi-GPU (L4 and L5 Async)

In [ ]:
# Multi-GPU benchmarks (2 A40 GPUs on Lighthouse)
if NUM_GPUS >= 2:
    print('='*60)
    print(f'Running Multi-GPU Benchmarks ({NUM_GPUS} GPUs)')
    print('='*60)
    
    multigpu_results = []
    
    for size in [2000, 4000, 8000, 16000, 32000]:
        print(f'\n--- Size: {size}x{size} ---')
        data_file = f'data/lowrank_{size}.bin'
        
        # L4: 2 GPUs (sync every iteration)
        print('MU L4 (2 GPUs, sync=1)...')
        time, error = run_mu('nmf_multigpu', data_file, RANK, MU_ITERS, '2')
        if time > 0:
            multigpu_results.append({'size': size, 'method': 'mu_l4_2gpu', 
                                    'time_ms': time, 'iterations': MU_ITERS, 'error': error})
            print(f'  Time: {time:.1f} ms, Error: {error:.4e}')
        
        # L5: Async with sync_interval=5 (default)
        print('MU L5 Async (2 GPUs, sync=5)...')
        time, error = run_mu('nmf_async_multigpu', data_file, RANK, MU_ITERS, '2 5')
        if time > 0:
            multigpu_results.append({'size': size, 'method': 'mu_l5_sync5', 
                                    'time_ms': time, 'iterations': MU_ITERS, 'error': error})
            print(f'  Time: {time:.1f} ms, Error: {error:.4e}')
        
        # L5: Async with sync_interval=10
        print('MU L5 Async (2 GPUs, sync=10)...')
        time, error = run_mu('nmf_async_multigpu', data_file, RANK, MU_ITERS, '2 10')
        if time > 0:
            multigpu_results.append({'size': size, 'method': 'mu_l5_sync10', 
                                    'time_ms': time, 'iterations': MU_ITERS, 'error': error})
            print(f'  Time: {time:.1f} ms, Error: {error:.4e}')
    
    if multigpu_results:
        multigpu_df = pd.DataFrame(multigpu_results)
        mu_df = pd.concat([mu_df, multigpu_df], ignore_index=True)
        mu_df.to_csv('../results/mu_timing.csv', index=False)
        print('\nUpdated: results/mu_timing.csv')
else:
    print(f'Only {NUM_GPUS} GPU detected. Multi-GPU tests skipped.')
    print('Request 2+ GPUs with: #SBATCH --gres=gpu:a40:2')

## 4. Scaling Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

colors = {
    'mu_naive': '#1f77b4', 'mu_l2_memory': '#2ca02c', 
    'mu_l3_compute': '#9467bd', 'mu_l4_2gpu': '#d62728',
    'mu_l5_sync5': '#ff7f0e', 'mu_l5_sync10': '#8c564b'
}
labels = {
    'mu_naive': 'MU L1 Naive', 'mu_l2_memory': 'MU L2 cuBLAS',
    'mu_l3_compute': 'MU L3 cuBLAS+ILP', 'mu_l4_2gpu': 'MU L4 (2 GPU, sync=1)',
    'mu_l5_sync5': 'MU L5 (2 GPU, sync=5)', 'mu_l5_sync10': 'MU L5 (2 GPU, sync=10)'
}
markers = {'mu_naive': 'o', 'mu_l2_memory': 's', 'mu_l3_compute': '^', 
           'mu_l4_2gpu': 'D', 'mu_l5_sync5': 'p', 'mu_l5_sync10': 'h'}

for method in mu_df['method'].unique():
    data = mu_df[mu_df['method'] == method].sort_values('size')
    ax.plot(data['size'], data['time_ms'], 
            marker=markers.get(method, 'o'), color=colors.get(method, 'gray'),
            label=labels.get(method, method), linewidth=2, markersize=8)

ax.set_xlabel('Matrix Size (n x n)')
ax.set_ylabel('Total Time (ms) - Log Scale')
ax.set_title('MU Scaling: Time vs Matrix Size')
ax.set_yscale('log')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('../results/figures/mu_scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/mu_scaling.png')

## 5. Speedup Analysis

In [ ]:
# Calculate speedup vs Naive baseline
speedup_data = []

for size in mu_df['size'].unique():
    size_data = mu_df[mu_df['size'] == size]
    naive_time = size_data[size_data['method'] == 'mu_naive']['time_ms'].values
    
    if len(naive_time) > 0:
        baseline = naive_time[0]
        for _, row in size_data.iterrows():
            speedup_data.append({
                'size': size,
                'method': row['method'],
                'time_ms': row['time_ms'],
                'speedup': baseline / row['time_ms']
            })

speedup_df = pd.DataFrame(speedup_data)

print('Speedup vs MU Naive:')
pivot = speedup_df.pivot(index='method', columns='size', values='speedup')
print(pivot.round(2).to_string())

In [ ]:
# Speedup bar chart
fig, ax = plt.subplots(figsize=(14, 6))

methods = [m for m in ['mu_l2_memory', 'mu_l3_compute', 'mu_l4_2gpu', 'mu_l5_sync5', 'mu_l5_sync10'] 
           if m in speedup_df['method'].unique()]
x = np.arange(len(mu_df['size'].unique()))
width = 0.15

for i, method in enumerate(methods):
    data = speedup_df[speedup_df['method'] == method].sort_values('size')
    if len(data) > 0:
        ax.bar(x[:len(data)] + i*width, data['speedup'], width,
               label=labels.get(method, method), color=colors.get(method, 'gray'))

ax.set_xlabel('Matrix Size')
ax.set_ylabel('Speedup vs MU L1 Naive')
ax.set_title('MU Optimization Speedup')
ax.set_xticks(x + width * len(methods) / 2)
ax.set_xticklabels(sorted(mu_df['size'].unique()))
ax.legend()
ax.axhline(y=1, color='k', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/figures/mu_speedup.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/mu_speedup.png')

## 6. Roofline Analysis

In [ ]:
# GPU specs - Lighthouse cluster A40
PEAK_GFLOPS = 37400   # A40 FP32 (37.4 TFLOPS)
PEAK_BANDWIDTH = 696  # GB/s

def compute_nmf_flops(m, n, k):
    """Theoretical FLOPs per MU iteration."""
    flops = 2*k*k*m + 2*k*n*m + 2*k*n*k  # H update GEMMs
    flops += 2*k*k*n + 2*m*k*n + 2*m*k*k  # W update GEMMs
    flops += 4*(k*n + m*k)  # Element-wise
    return flops

def compute_nmf_bytes(m, n, k):
    """Memory movement per iteration (bytes)."""
    return m*n*4 + m*k*4*2 + k*n*4*2 + (k*k + k*n + m*k)*4

# Calculate roofline data
print('Roofline Analysis:')
print('='*70)

roofline_data = []
for _, row in mu_df.iterrows():
    size = row['size']
    flops = compute_nmf_flops(size, size, RANK) * row['iterations']
    bytes_moved = compute_nmf_bytes(size, size, RANK) * row['iterations']
    ai = flops / bytes_moved
    achieved_gflops = flops / (row['time_ms'] * 1e6)
    efficiency = achieved_gflops / PEAK_GFLOPS * 100
    
    roofline_data.append({
        'method': row['method'],
        'size': size,
        'arithmetic_intensity': ai,
        'achieved_gflops': achieved_gflops,
        'peak_efficiency_pct': efficiency
    })
    print(f"{row['method']:15} ({size}): AI={ai:.1f}, {achieved_gflops:.0f} GFLOPS ({efficiency:.1f}% peak)")

# Save roofline analysis
roofline_df = pd.DataFrame(roofline_data)
roofline_df.to_csv('../results/roofline_analysis.csv', index=False)
print(f'\nSaved: results/roofline_analysis.csv')

## 7. Summary Table

In [ ]:
print('='*70)
print('MU BENCHMARK SUMMARY')
print('='*70)

print('\n| Method | Size | Time (ms) | Speedup | Error |')
print('|--------|------|-----------|---------|-------|')

merged = mu_df.merge(speedup_df[['size', 'method', 'speedup']], on=['size', 'method'])
for _, row in merged.sort_values(['size', 'method']).iterrows():
    print(f"| {labels.get(row['method'], row['method'])} | {row['size']} | {row['time_ms']:.1f} | {row['speedup']:.2f}x | {row['error']:.4e} |")

print('\n### Key Findings')
print('1. L1→L2: Massive speedup from naive GEMM to cuBLAS (10-50x)')
print('2. L2→L3: Minimal improvement (cuBLAS dominates, Amdahl\'s Law)')
print('3. L4 Multi-GPU: Communication overhead exceeds compute savings (no NVLink)')
print('4. L5 Async: Reduced sync frequency improves speed but may affect accuracy')
print('5. Trade-off: sync=5 balances speed vs accuracy, sync=10 faster but less accurate')

## Done!

### Output Files
- `results/mu_timing.csv` - Timing data
- `results/figures/mu_scaling.png` - Scaling plot
- `results/figures/mu_speedup.png` - Speedup chart